# LeetCode 1849 — Splitting a String Into Descending Consecutive Values bằng Python

Link đề: https://leetcode.com/problems/splitting-a-string-into-descending-consecutive-values/ · Độ khó: **Medium** · Chủ đề: String, Backtracking, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def splitString(self, s: str) -> bool:
        ...
```

## 1. Mục tiêu học

1. Biểu diễn mọi cách **cắt chuỗi** thành các mảnh bằng **bitmask** các vị trí cắt (`2^{n−1}` cách) và viết vét cạn.
2. Viết **quay lui** (backtracking): chọn mảnh đầu, rồi mỗi mảnh sau phải có giá trị **đúng bằng** mảnh trước trừ 1 — cắt tỉa rất mạnh.
3. Nhận ra rằng khi biết giá trị mảnh đầu, toàn bộ cách cắt là **duy nhất** (nếu tồn tại) ⇒ kiểm tra **tham lam** `O(n)` cho mỗi mảnh đầu, tổng `O(n²)`.
4. Xử lý đúng **số 0 ở đầu** (leading zeros) và trường hợp giá trị **0** (chỉ có thể là mảnh cuối).
5. Kiểm thử bằng chuỗi ngẫu nhiên **có cấu trúc** (ghép các số giảm dần với số 0 chèn ngẫu nhiên) để có nhiều đáp án `True`.

## 2. Đề bài & ví dụ

Cho chuỗi `s` chỉ gồm chữ số. Kiểm tra xem có thể cắt `s` thành **hai hoặc nhiều** chuỗi con khác rỗng sao cho **giá trị số** của các mảnh **giảm dần** và hai mảnh kề nhau **chênh nhau đúng 1**.

- `"0090089"` cắt thành `["0090", "089"]` ⇒ `[90, 89]` ✓.
- `"001"` có các cách `["0","01"]`, `["00","1"]`, `["0","0","1"]` ⇒ `[0,1]`, `[0,1]`, `[0,0,1]` — đều không giảm ⇒ ✗.

**Ví dụ 1:** `s = "1234"` → `False`.

**Ví dụ 2:** `s = "050043"` → `True` (`["05","004","3"]` ⇒ `[5,4,3]`).

**Ví dụ 3:** `s = "9080701"` → `False`.

**Ràng buộc:** `1 ≤ s.length ≤ 20`; `s` chỉ gồm chữ số.

In [1]:
import random
import time
from functools import lru_cache
from typing import List

examples = [("1234", False), ("050043", True), ("9080701", False)]
for s, exp in examples:
    print(f"s={s!r} -> kỳ vọng {exp}")

s='1234' -> kỳ vọng False
s='050043' -> kỳ vọng True
s='9080701' -> kỳ vọng False


## 3. Kiến thức nền

### 3.1 Giá trị số của chuỗi chữ số và số 0 ở đầu

`int("004") = 4`, `int("000") = 0`. Số 0 ở đầu không đổi giá trị. Cẩn thận: chuỗi 20 chữ số có giá trị tới `10²⁰ − 1` — Python không tràn, nhưng C++/Java phải lo (dùng `long` và dừng sớm).

In [2]:
print(int("004"), int("000"), int("0090"), int("99999999999999999999"))

4 0 90 99999999999999999999


### 3.2 Bitmask các vị trí cắt

Chuỗi dài `n` có `n − 1` "khe" giữa các ký tự. Mỗi số `mask ∈ [0, 2^{n−1})` chọn những khe được cắt (bit `i` bật ⇒ cắt sau ký tự `i`).

In [3]:
def split_by_mask(s, mask):
    parts, start = [], 0
    for i in range(len(s) - 1):
        if mask >> i & 1:
            parts.append(s[start:i + 1]); start = i + 1
    parts.append(s[start:])
    return parts

s = "0543"
for mask in range(1 << (len(s) - 1)):
    print(f"mask={mask:03b} -> {split_by_mask(s, mask)}")

mask=000 -> ['0543']
mask=001 -> ['0', '543']
mask=010 -> ['05', '43']
mask=011 -> ['0', '5', '43']
mask=100 -> ['054', '3']
mask=101 -> ['0', '54', '3']
mask=110 -> ['05', '4', '3']
mask=111 -> ['0', '5', '4', '3']


### 3.3 Kiểm tra một dãy giá trị hợp lệ

In [4]:
def ok_values(vals):
    return len(vals) >= 2 and all(vals[i] - vals[i + 1] == 1 for i in range(len(vals) - 1))

print(ok_values([5, 4, 3]), ok_values([90, 89]), ok_values([0, 0, 1]), ok_values([7]))

True True False False


### 3.4 Quay lui (backtracking) với cắt tỉa

`dfs(i, prev)`: đã xử lý `s[:i]`, mảnh cuối có giá trị `prev`. Thử mọi mảnh tiếp theo `s[i:j+1]`; chỉ đi tiếp nếu giá trị đúng `prev − 1`. Có thể **dừng sớm** khi giá trị mảnh đang xây đã vượt `prev − 1` (thêm chữ số chỉ làm số lớn hơn).

## 4. Ý tưởng nền tảng / trực giác

**Quan sát then chốt:** nếu biết giá trị mảnh **đầu tiên** `x`, các mảnh sau **bắt buộc** là `x−1, x−2, …`. Và với mỗi giá trị cần tìm `v > 0`, mảnh bắt đầu tại vị trí `i` phải có dạng `0…0` + `str(v)`; số chữ số 0 ở đầu bằng đúng số chữ số 0 liên tiếp tại `i` (vì `str(v)` bắt đầu bằng chữ số khác 0) ⇒ mảnh được **xác định duy nhất**. Riêng `v = 0`: mảnh phải là toàn bộ phần còn lại và chỉ gồm `0` (sau 0 không còn số `−1`).

Tính tay **Ví dụ 2** `"050043"`, thử mảnh đầu:
- `"0"` (x = 0): mảnh sau phải là −1 ⇒ ✗.
- `"05"` (x = 5): cần 4 tại vị trí 2: `"004"` ✓; cần 3 tại vị trí 5: `"3"` ✓; hết chuỗi ⇒ **True**.

**Ví dụ 3** `"9080701"`: mảnh đầu `"9"` ⇒ cần 8: vị trí 1 là `"0"`… giá trị `"08" = 8` ✓ ⇒ cần 7: `"07"` ✓ ⇒ cần 6: còn `"01"` = 1 ✗. Mảnh đầu `"90"` ⇒ cần 89 … ✗. ⇒ False.

In [5]:
s = "050043"
for L in range(1, len(s)):
    print(f"mảnh đầu {s[:L]!r} = {int(s[:L])} -> dãy cần: {[int(s[:L]) - t for t in range(1, 4)]}")

mảnh đầu '0' = 0 -> dãy cần: [-1, -2, -3]
mảnh đầu '05' = 5 -> dãy cần: [4, 3, 2]
mảnh đầu '050' = 50 -> dãy cần: [49, 48, 47]
mảnh đầu '0500' = 500 -> dãy cần: [499, 498, 497]
mảnh đầu '05004' = 5004 -> dãy cần: [5003, 5002, 5001]


## 5. Cách 1 — Brute Force / baseline: thử mọi cách cắt bằng bitmask

In [6]:
def split_bruteforce(s):
    n = len(s)
    for mask in range(1, 1 << (n - 1)):               # mask = 0 là không cắt (chỉ 1 mảnh) -> bỏ
        if ok_values([int(p) for p in split_by_mask(s, mask)]):
            return True
    return False

for s, exp in examples:
    got = split_bruteforce(s); print(got, "kỳ vọng", exp); assert got == exp

False kỳ vọng False
True kỳ vọng True
False kỳ vọng False


**Độ phức tạp:** `2^{n−1} · n` ≈ `5·10⁵ · 20 = 10⁷` thao tác với `n = 20` — chạy được nhưng chậm (vài giây trong Python).

**Nút thắt:** thử cả những cách cắt mà mảnh thứ hai đã sai — không cắt tỉa gì.

In [7]:
t = time.perf_counter(); split_bruteforce("9" * 16); print(f"vét cạn n=16: {(time.perf_counter()-t)*1000:.0f} ms (n=20 chậm hơn ~16 lần)")

vét cạn n=16: 54 ms (n=20 chậm hơn ~16 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (cắt tỉa theo giá trị):** khi đã biết `prev`, mảnh tiếp theo phải có giá trị `prev − 1`. Xây mảnh từng chữ số; khi giá trị vượt `prev − 1` thì dừng (thêm chữ số chỉ tăng giá trị, trừ khi đang toàn 0).

**Insight 2 (mảnh đầu quyết định tất cả):** cố định mảnh đầu `s[:L]` (`L = 1..n−1`, phải chừa ít nhất 1 ký tự). Với mỗi giá trị cần tìm, mảnh tiếp theo xác định duy nhất ⇒ kiểm tra bằng một lượt quét `O(n)` ⇒ tổng `O(n²)`.

**Quy tắc kiểm tra tham lam (cần `v`, đang ở vị trí `i`):**
- **Nếu** `v == 0` **thì** phần còn lại `s[i:]` phải khác rỗng và toàn `'0'`.
- **Ngược lại**: bỏ qua các `'0'` liên tiếp, rồi phần tiếp theo phải bắt đầu bằng `str(v)`; nhảy `i` qua đó, cần `v − 1`.
- Thành công khi `i == n` sau ít nhất một mảnh sau mảnh đầu.

**Invariant:** tại mỗi bước, `s[:i]` đã được cắt hợp lệ thành dãy `x, x−1, …, v+1`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quay lui với cắt tỉa

In [8]:
def split_backtrack(s):
    n = len(s)
    def dfs(i, prev, count):
        if i == n:
            return count >= 2
        val = 0
        for j in range(i, n):
            val = val * 10 + int(s[j])
            if prev is not None and val > prev - 1:
                break                                  # giá trị chỉ tăng thêm -> dừng
            if (prev is None or val == prev - 1) and dfs(j + 1, val, count + 1):
                return True
        return False
    return dfs(0, None, 0)

print([split_backtrack(s) for s, _ in examples])

[False, True, False]


Lưu ý điều kiện `break`: nếu `val > prev − 1` thì thêm chữ số nữa (`val·10 + d ≥ val`) cũng không giảm được ⇒ dừng an toàn. Khi `val` đang là 0 (toàn số 0) thì `val ≤ prev − 1` vẫn tiếp tục — đúng với leading zeros.

**Code nộp LeetCode (Cách 2):**

In [9]:
class SolutionBacktrack:
    def splitString(self, s: str) -> bool:
        n = len(s)

        def dfs(i: int, prev: int) -> bool:
            if i == n:
                return True
            val = 0
            for j in range(i, n):
                val = val * 10 + ord(s[j]) - 48
                if val > prev - 1:
                    break
                if val == prev - 1 and dfs(j + 1, val):
                    return True
            return False

        first = 0
        for L in range(1, n):                          # mảnh đầu dài L, chừa ít nhất 1 ký tự
            first = first * 10 + ord(s[L - 1]) - 48
            if dfs(L, first):
                return True
        return False

print([SolutionBacktrack().splitString(s) for s, _ in examples])

[False, True, False]


**Độ phức tạp Cách 2:** tệ nhất khó đánh giá chặt, nhưng nhờ cắt tỉa (với mỗi `prev`, chỉ một độ dài mảnh có giá trị đúng `prev − 1` trừ phần số 0), thực tế gần `O(n²)`. Độ sâu đệ quy `≤ n = 20`.

### 7.2 Cách 3 — Thử mảnh đầu + kiểm tra tham lam `O(n)`

In [10]:
def follows(s, i, v):
    """Từ vị trí i, phần còn lại có cắt được thành v, v-1, ..., (≥ 1 mảnh) không? (cắt duy nhất)"""
    n = len(s)
    while True:
        if v < 0:
            return False
        if v == 0:
            return i < n and set(s[i:]) == {"0"}
        while i < n and s[i] == "0":                   # leading zeros của mảnh giá trị v
            i += 1
        t = str(v)
        if not s.startswith(t, i):
            return False
        i += len(t)
        if i == n:
            return True
        v -= 1

def split_greedy(s):
    n = len(s)
    for L in range(1, n):
        if follows(s, L, int(s[:L]) - 1):
            return True
    return False

print([split_greedy(s) for s, _ in examples])

[False, True, False]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** `n − 1` mảnh đầu × `O(n)` kiểm tra (mỗi chữ số được đọc một lần) ⇒ `O(n²)` = 400 thao tác với `n = 20`.

### 7.3 Cách 4 — Quy hoạch động có ghi nhớ trên `(i, prev)`

Cùng định nghĩa với Cách 2 nhưng `@lru_cache` trên `(i, prev)` — tránh tính lại khi nhiều nhánh gặp cùng trạng thái. Với bài này nhánh trùng hiếm (vì cách cắt gần như duy nhất), nhưng đây là khuôn mẫu chung cho các bài "cắt chuỗi thành dãy có quy luật".

In [11]:
def split_memo(s):
    n = len(s)
    @lru_cache(maxsize=None)
    def can(i, prev):
        if i == n:
            return True
        val = 0
        for j in range(i, n):
            val = val * 10 + int(s[j])
            if val > prev - 1: break
            if val == prev - 1 and can(j + 1, val):
                return True
        return False
    return any(can(L, int(s[:L])) for L in range(1, n))

print([split_memo(s) for s, _ in examples])

[False, True, False]


**Độ phức tạp Cách 4:** số trạng thái `(i, prev)` thực sự đi tới là `O(n²)`, mỗi trạng thái `O(n)` ⇒ `O(n³)` tệ nhất — vẫn rất nhỏ.

## 8. TRACE từng bước

Trace Cách 3 trên `"0090089"` và `"9080701"`:

In [12]:
def trace_greedy(s):
    n = len(s)
    for L in range(1, n):
        x = int(s[:L]); i, v = L, x - 1; path = [s[:L]]
        status = None
        while True:
            if v < 0:
                status = "cần số âm -> thất bại"; break
            if v == 0:
                ok = i < n and set(s[i:]) == {"0"}
                if ok: path.append(s[i:])
                status = "phần còn lại toàn 0 -> THÀNH CÔNG" if ok else "cần 0 nhưng phần còn lại không toàn 0"; break
            j = i
            while j < n and s[j] == "0": j += 1
            t = str(v)
            if not s.startswith(t, j):
                status = f"tại vị trí {i} cần {v} nhưng gặp {s[i:]!r}"; break
            path.append(s[i:j + len(t)]); i = j + len(t)
            if i == n:
                status = "hết chuỗi -> THÀNH CÔNG"; break
            v -= 1
        print(f"mảnh đầu {s[:L]!r} ({x}): {path} -> {status}")
        if "THÀNH CÔNG" in status:
            return True
    return False

assert trace_greedy("0090089") is True
print()
assert trace_greedy("9080701") is False

mảnh đầu '0' (0): ['0'] -> cần số âm -> thất bại
mảnh đầu '00' (0): ['00'] -> cần số âm -> thất bại
mảnh đầu '009' (9): ['009', '008'] -> tại vị trí 6 cần 7 nhưng gặp '9'
mảnh đầu '0090' (90): ['0090', '089'] -> hết chuỗi -> THÀNH CÔNG

mảnh đầu '9' (9): ['9', '08', '07'] -> tại vị trí 5 cần 6 nhưng gặp '01'
mảnh đầu '90' (90): ['90'] -> tại vị trí 2 cần 89 nhưng gặp '80701'
mảnh đầu '908' (908): ['908'] -> tại vị trí 3 cần 907 nhưng gặp '0701'
mảnh đầu '9080' (9080): ['9080'] -> tại vị trí 4 cần 9079 nhưng gặp '701'
mảnh đầu '90807' (90807): ['90807'] -> tại vị trí 5 cần 90806 nhưng gặp '01'
mảnh đầu '908070' (908070): ['908070'] -> tại vị trí 6 cần 908069 nhưng gặp '1'


Tóm tắt trace cho `"0090089"` (chỉ số 0..6):

| mảnh đầu | giá trị | các bước tiếp theo | kết quả |
|---|---|---|---|
| `"0"` | 0 | cần −1 | ✗ |
| `"00"` | 0 | cần −1 | ✗ |
| `"009"` | 9 | cần 8: bỏ `"0"` ở vị trí 3–4, khớp `"8"` ⇒ mảnh `"008"`; cần 7: còn `"9"` | ✗ |
| `"0090"` | 90 | cần 89: bỏ `"0"` ở vị trí 4, khớp `"89"` ⇒ mảnh `"089"`, hết chuỗi | **✓** |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — thử mảnh đầu + kiểm tra tham lam.

In [13]:
class Solution:
    def splitString(self, s: str) -> bool:
        n = len(s)

        def follows(i: int, v: int) -> bool:
            while v >= 0:
                if v == 0:
                    return i < n and s[i:].count("0") == n - i
                while i < n and s[i] == "0":
                    i += 1
                t = str(v)
                if not s.startswith(t, i):
                    return False
                i += len(t)
                if i == n:
                    return True
                v -= 1
            return False

        for L in range(1, n):
            if follows(L, int(s[:L]) - 1):
                return True
        return False

for s, exp in examples:
    got = Solution().splitString(s); print(got, exp); assert got == exp

False False
True True
False False


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (cắt duy nhất khi biết giá trị cần).** Đứng ở vị trí `i`, cần một mảnh có giá trị `v > 0`. Mảnh là `s[i..j]` với `int(s[i..j]) = v` ⇔ `s[i..j]` = (một số chữ số 0) + `str(v)`. Vì `str(v)` bắt đầu bằng chữ số khác 0, phần "số 0" phải là **toàn bộ** dãy 0 liên tiếp tại `i` (nếu dừng sớm hơn, ký tự kế tiếp là 0 ≠ chữ số đầu của `str(v)`). Vậy `j` xác định duy nhất. Với `v = 0`: mảnh là toàn số 0; nếu sau nó còn ký tự thì cần mảnh `−1` — không tồn tại ⇒ mảnh phải kéo tới hết chuỗi.

**Bổ đề 2.** Mọi cách cắt hợp lệ có mảnh đầu `s[:L]` với `1 ≤ L ≤ n − 1`, và các mảnh sau là `x−1, x−2, …` với `x = int(s[:L])`. Theo Bổ đề 1, nếu cách cắt như vậy tồn tại thì `follows(L, x − 1)` tìm ra đúng nó và trả `True`; ngược lại trả `False`.

⇒ Thuật toán trả `True` ⇔ tồn tại `L` có cách cắt hợp lệ ⇔ đáp án `True`. ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s) ≤ 20`.

| Cách | Time | Space |
|---|---|---|
| 1. Bitmask mọi cách cắt | `O(2^{n−1} · n)` | `O(n)` |
| 2. Quay lui + cắt tỉa | thực tế `≈ O(n²)` | `O(n)` (đệ quy) |
| 3. Mảnh đầu + tham lam | `O(n²)` | `O(n)` |
| 4. Ghi nhớ `(i, prev)` | `O(n³)` tệ nhất | `O(n²)` |

In [14]:
hard = ["9" * 20, "1" * 20, "0" * 19 + "1", "10" * 10, "9876543210" * 2]
for name, f in [("backtrack", SolutionBacktrack().splitString), ("greedy", Solution().splitString), ("memo", split_memo)]:
    t = time.perf_counter()
    r = [f(x) for x in hard * 50]
    print(f"{name:<10} {(time.perf_counter()-t)*1000:7.1f} ms cho {len(hard) * 50} chuỗi | kết quả {r[:5]}")

backtrack      2.6 ms cho 250 chuỗi | kết quả [False, False, False, False, False]
greedy         1.1 ms cho 250 chuỗi | kết quả [False, False, False, False, False]
memo           4.7 ms cho 250 chuỗi | kết quả [False, False, False, False, False]


## 12. Bộ test

Chuỗi ngẫu nhiên thuần hầu như cho `False`; ta tạo thêm chuỗi **có cấu trúc**: chọn `x` ngẫu nhiên, ghép `x, x−1, …` (mỗi số có thể chèn 0 ở đầu), đôi khi làm hỏng một chữ số. Đối chiếu với vét cạn bitmask (giới hạn `n ≤ 14` để chạy nhanh).

In [15]:
ALL = {"backtrack": split_backtrack, "SolBacktrack": SolutionBacktrack().splitString, "greedy": split_greedy,
       "memo": split_memo, "submit": Solution().splitString}
def run_all(s, brute=True):
    res = {k: f(s) for k, f in ALL.items()}
    if brute:
        res["brute"] = split_bruteforce(s)
    assert len(set(res.values())) == 1, (s, res)
    return res["submit"]

for s, exp in examples:
    assert run_all(s) == exp
edge = [("1", False), ("10", True), ("00", False), ("0090089", True), ("001", False), ("21", True), ("200100", True),
        ("4321", True), ("9876543210", True), ("100099", True), ("10009998", True), ("1000", True), ("0000", False), ("20", False)]
for s, exp in edge:
    assert run_all(s) == exp, s
def structured(rng):
    x = rng.randint(0, 120); k = rng.randint(2, 4); parts = []
    for t in range(k):
        v = x - t
        if v < 0: break
        parts.append("0" * rng.randint(0, 2) + str(v))
    s = "".join(parts)
    if rng.random() < 0.3 and s:
        j = rng.randrange(len(s)); s = s[:j] + str(rng.randint(0, 9)) + s[j + 1:]
    return s[:14] or "0"
rng = random.Random(18490)
cnt = {True: 0, False: 0}
for _ in range(500):
    s = structured(rng); cnt[run_all(s)] += 1
for _ in range(300):
    s = "".join(rng.choice("0123456789") for _ in range(rng.randint(1, 12))); run_all(s)
for _ in range(200):
    s = structured(rng) + "".join(rng.choice("0123") for _ in range(rng.randint(0, 6)))
    run_all(s[:20], brute=len(s) <= 14)
print("Ví dụ, edge case, 1000 test (có cấu trúc + ngẫu nhiên) OK | phân bố có cấu trúc:", cnt)

Ví dụ, edge case, 1000 test (có cấu trúc + ngẫu nhiên) OK | phân bố có cấu trúc: {True: 349, False: 151}


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho phép chỉ **một** mảnh (không cắt)

In [16]:
def wrong_one_piece(s):
    n = len(s)
    for L in range(1, n + 1):                          # SAI: L = n nghĩa là không cắt
        if L == n: return True
        if follows(s, L, int(s[:L]) - 1): return True
    return False

print("sai :", wrong_one_piece("1234"), "| đúng:", Solution().splitString("1234"))

sai : True | đúng: False


**Vì sao sai:** đề yêu cầu **ít nhất hai** mảnh. **Sửa:** mảnh đầu dài tối đa `n − 1`.

### Lỗi 2 — Bỏ qua số 0 ở đầu sai cách (so sánh chuỗi trực tiếp với `str(v)`)

In [17]:
def wrong_no_leading_zero(s):
    n = len(s)
    for L in range(1, n):
        i, v, ok = L, int(s[:L]) - 1, False
        while v >= 0:
            t = str(v)
            if not s.startswith(t, i): break                    # SAI: không cho phép "004"
            i += len(t)
            if i == n: ok = True; break
            v -= 1
        if ok: return True
    return False

s, exp = examples[1]
print("sai :", wrong_no_leading_zero(s), "| đúng:", exp)

sai : False | đúng: True


**Vì sao sai:** `"004"` có giá trị 4 — số 0 ở đầu là hợp lệ. **Sửa:** bỏ qua các `'0'` liên tiếp trước khi khớp `str(v)`.

### Lỗi 3 — Coi `0` có thể đứng giữa (cho phép tiếp `−1`?) hoặc chấp nhận `[0, 0]`

In [18]:
def wrong_equal_ok(s):
    n = len(s)
    for mask in range(1, 1 << (n - 1)):
        vals = [int(p) for p in split_by_mask(s, mask)]
        if all(vals[i] - vals[i + 1] in (0, 1) for i in range(len(vals) - 1)):   # SAI: cho phép bằng nhau
            return True
    return False

print("sai :", wrong_equal_ok("00"), "| đúng:", Solution().splitString("00"))
print("sai :", wrong_equal_ok("5544"), "| đúng:", Solution().splitString("5544"))

sai : True | đúng: False
sai : True | đúng: False


**Vì sao sai:** đề yêu cầu **giảm dần**, chênh **đúng 1** — không được bằng nhau. **Sửa:** `vals[i] − vals[i+1] == 1`.

### Lỗi 4 — Quay lui dừng sớm (`break`) khi giá trị đang là 0

In [19]:
def wrong_break_zero(s):
    n = len(s)
    def dfs(i, prev):
        if i == n: return True
        val = 0
        for j in range(i, n):
            val = val * 10 + int(s[j])
            if val != prev - 1:
                break                                  # SAI: "0" chưa phải là giá trị cuối của "004"
            if dfs(j + 1, val): return True
        return False
    return any(dfs(L, int(s[:L])) for L in range(1, n))

s, exp = examples[1]
print("sai :", wrong_break_zero(s), "| đúng:", exp)

sai : False | đúng: True


**Vì sao sai:** đọc tới `"0"` thì giá trị (0) chưa bằng 4, nhưng đọc thêm `"04"`, `"004"` mới ra 4. Chỉ được dừng khi giá trị **vượt** `prev − 1`. **Sửa:** `if val > prev − 1: break`.

### Lỗi 5 — Ép kiểu số thực hoặc so sánh chuỗi thay cho số

In [20]:
def wrong_string_compare(s):
    n = len(s)
    for L in range(1, n):
        a, b = s[:L], s[L:]
        if a > b and len(b) > 0 and int(a) - int(b) == 1:     # chỉ thử đúng 2 mảnh (SAI) + so sánh chuỗi
            return True
    return False

s, exp = examples[1]
print("sai :", wrong_string_compare(s), "| đúng:", exp)

sai : False | đúng: True


**Vì sao sai:** chỉ xét cách cắt **hai** mảnh (bỏ sót 3 mảnh trở lên như `[5,4,3]`), và so sánh chuỗi `"05" > "0043"` không phản ánh giá trị số. **Sửa:** thử mọi số mảnh, so sánh bằng `int`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `n ≤ 20` + "cắt chuỗi" ⇒ vét cạn `2^{n−1}` hoặc quay lui đều được; quay lui có cắt tỉa là lựa chọn an toàn.
- Dãy có quy luật cố định (giảm 1, cấp số cộng, Fibonacci…) ⇒ **mảnh đầu (hoặc hai mảnh đầu) quyết định cả dãy** ⇒ thử mảnh đầu + kiểm tra tham lam.
- Leading zeros: giá trị không đổi; nhớ trường hợp `0` đặc biệt.
- Quay lui trên chữ số: dừng khi giá trị **vượt** mục tiêu, không phải khi **khác** mục tiêu.

## 15. Biến thể của bài

1. **842 Split Array into Fibonacci Sequence** và **306 Additive Number**: hai mảnh đầu quyết định dãy; cấm leading zeros (trừ chính số 0).
2. **Dãy tăng dần chênh 1** (`x, x+1, …`): đối xứng (Bài tập 5).
3. **Chênh lệch `d` tùy ý nhưng cố định**: thử hai mảnh đầu để biết `d`.
4. **Đếm số cách cắt** hợp lệ: với quy luật giảm 1, mỗi mảnh đầu cho tối đa một cách ⇒ đếm số `L` thành công (Bài tập 4).
5. **1593 Split a String Into the Max Number of Unique Substrings**: quay lui cắt chuỗi với ràng buộc khác.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bitmask | Mọi cách cắt | `O(2^n · n)` | `O(n)` | Kiểm chứng |
| 2. Quay lui + cắt tỉa | Mảnh sau = mảnh trước − 1 | `≈ O(n²)` | `O(n)` | Tổng quát, dễ viết |
| 3. Mảnh đầu + tham lam | Cắt duy nhất | `O(n²)` | `O(n)` | **Bản nộp** |
| 4. Ghi nhớ | `lru_cache(i, prev)` | `O(n³)` | `O(n²)` | Khuôn mẫu cho bài cắt chuỗi khác |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `"10"`; (b) `"0"`; (c) `"3020100"`; (d) `"99989796"`.

**Bài 2 (tính tay):** Với `"100099"`: thử từng mảnh đầu và ghi các mảnh sau theo quy tắc tham lam.

**Bài 3 (điền TODO):** Hoàn thiện `piece_value(s, i, j)` trả `int(s[i:j+1])` mà không dùng `int()` (cộng dồn chữ số).

**Bài 4 (tự viết lại):** Viết `all_splits(s)` trả **danh sách mọi cách cắt** hợp lệ (mỗi cách là list chuỗi), dùng mảnh đầu + tham lam.

**Bài 5 (mini challenge):** Viết `split_ascending(s)` kiểm tra cắt được thành **≥ 2** mảnh **tăng dần, chênh đúng 1**. Lưu ý: giá trị tăng nên số 0 chỉ có thể là mảnh **đầu**.

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def piece_value_todo(s, i, j):
    v = 0
    for t in range(i, j + 1):
        # TODO: v = v * 10 + (chữ số s[t])
        pass
    return v

In [22]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def all_splits(s):
    # TODO: với mỗi L, chạy tham lam nhưng ghi lại các mảnh; thành công thì thêm vào kết quả
    pass

def split_ascending(s):
    # TODO: thử mảnh đầu; mảnh sau cần v+1: bỏ qua số 0 đầu rồi khớp str(v+1)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: thử mảnh đầu. Mức 2: tham lam. Mức 3: (a) `True` (`[1,0]`); (b) `False` (một ký tự); (c) `True`: mảnh đầu `"3"`, cần 2 ⇒ `"02"`, cần 1 ⇒ `"01"`, cần 0 ⇒ phần còn lại `"00"` toàn 0 ✓ ⇒ `[3, 2, 1, 0]`; (d) `True` (`[99, 98, 97, 96]`).

**Bài 2.** Mức 1: `"1"` ⇒ cần 0: còn `"00099"` không toàn 0 ✗. Mức 2: `"10"` ⇒ cần 9: bỏ `"00"`, gặp `"99"` bắt đầu bằng "9" ✓ (`"009"`), cần 8: còn `"9"` ✗. Mức 3: `"100"` ⇒ cần 99: `"099"` ✓ hết chuỗi ⇒ `True`.

**Bài 3.** Mức 1: `ord(s[t]) − 48`. Mức 2: nhân 10 rồi cộng. Mức 3: so với `int`.

**Bài 4.** Mức 1: lưu `path`. Mức 2: thêm `path` khi thành công. Mức 3: số cách = số `L` thành công; đối chiếu vét cạn bitmask.

**Bài 5.** Mức 1: mảnh sau cần `v + 1 > 0` ⇒ không bao giờ là 0. Mức 2: mảnh đầu có thể là `"0…0"` (giá trị 0). Mức 3: đối chiếu vét cạn bitmask với điều kiện tăng.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2, 3
b1 = [Solution().splitString(x) for x in ["10", "0", "3020100", "99989796"]]
print("Bài 1:", b1); assert b1 == [True, False, True, True]
assert b1 == [split_bruteforce(x) for x in ["10", "0", "3020100", "99989796"]]
print("Bài 2:"); assert trace_greedy("100099") is True
def piece_value_todo(s, i, j):
    v = 0
    for t in range(i, j + 1):
        v = v * 10 + ord(s[t]) - 48
    return v
assert all(piece_value_todo("0050043", i, j) == int("0050043"[i:j + 1]) for i in range(7) for j in range(i, 7))
print("Bài 3: OK")

Bài 1: [True, False, True, True]
Bài 2:
mảnh đầu '1' (1): ['1'] -> cần 0 nhưng phần còn lại không toàn 0
mảnh đầu '10' (10): ['10', '009'] -> tại vị trí 5 cần 8 nhưng gặp '9'
mảnh đầu '100' (100): ['100', '099'] -> hết chuỗi -> THÀNH CÔNG
Bài 3: OK


In [24]:
# Bài 4, 5
def all_splits(s):
    n = len(s); res = []
    for L in range(1, n):
        i, v, path = L, int(s[:L]) - 1, [s[:L]]
        while v >= 0:
            if v == 0:
                if i < n and set(s[i:]) == {"0"}: path.append(s[i:]); res.append(path)
                break
            j = i
            while j < n and s[j] == "0": j += 1
            t = str(v)
            if not s.startswith(t, j): break
            path.append(s[i:j + len(t)]); i = j + len(t)
            if i == n: res.append(path); break
            v -= 1
    return res

def split_ascending(s):
    n = len(s)
    for L in range(1, n):
        i, v = L, int(s[:L]) + 1
        while True:
            j = i
            while j < n and s[j] == "0": j += 1
            t = str(v)
            if not s.startswith(t, j): break
            i = j + len(t)
            if i == n: return True
            v += 1
    return False

def ascending_brute(s):
    n = len(s)
    for mask in range(1, 1 << (n - 1)):
        vals = [int(p) for p in split_by_mask(s, mask)]
        if all(vals[k + 1] - vals[k] == 1 for k in range(len(vals) - 1)):
            return True
    return False

rng = random.Random(9)
for _ in range(400):
    s = structured(rng)
    brute_all = sorted(split_by_mask(s, m) for m in range(1, 1 << (len(s) - 1)) if ok_values([int(p) for p in split_by_mask(s, m)])) if len(s) > 1 else []
    assert sorted(all_splits(s)) == brute_all, s
    a = "".join(rng.choice("0123456789") for _ in range(rng.randint(1, 10)))
    x = rng.randint(0, 50); b = "".join("0" * rng.randint(0, 1) + str(x + t) for t in range(rng.randint(2, 3)))[:13]
    for w in (a, b):
        assert split_ascending(w) == ascending_brute(w), w
print("Bài 4, 5: OK |", all_splits("0090089"), all_splits("3020100"), split_ascending("0102"), split_ascending("91011"))

Bài 4, 5: OK | [['0090', '089']] [['3', '02', '01', '00']] True True


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "mảnh đầu quyết định dãy":**

```python
for L in range(1, n):                 # chừa ít nhất 1 ký tự
    i, v = L, int(s[:L]) - 1
    while v >= 0:
        if v == 0: ok = (i < n and s[i:] toàn '0'); break
        bỏ qua các '0' tại i; nếu s không bắt đầu bằng str(v) tại đó: thất bại
        i += len(str(v)); nếu i == n: THÀNH CÔNG
        v -= 1
```

**Checklist:**
- [ ] Ít nhất **hai** mảnh (mảnh đầu dài `≤ n − 1`)?
- [ ] Cho phép **số 0 ở đầu** mỗi mảnh?
- [ ] Chênh **đúng 1**, không bằng nhau?
- [ ] Giá trị 0 chỉ là mảnh **cuối** và phần còn lại toàn `0`?
- [ ] Quay lui: `break` khi giá trị **vượt** mục tiêu (không phải khi khác)?
- [ ] Đã test `"10"`, `"00"`, `"001"`, `"0090089"`, `"1"`, chuỗi 20 chữ số?